<a href="https://colab.research.google.com/github/rkvirdi/text_classification_bert/blob/main/text_classification_bert_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers datasets torch evaluate -q

In [3]:
from google.colab import userdata
from huggingface_hub import login
import numpy as np
huggingfacekey = userdata.get('huggingfaceapi')
login(token=huggingfacekey)

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate

In [5]:
dataset = load_dataset('ag_news')
# Print the dataset structure
print(dataset)
rand = np.random.randint(10000,size=10)
print(rand)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
[6811 4209 2408 8729 3327 2568 6896 4741 8547  444]


In [6]:
import random
# Sample 200 random training examples
train_samples = random.sample(range(len(dataset['train'])), 200)
train_dataset = dataset['train'].select(train_samples)

# Sample 20 random test examples
test_samples = random.sample(range(len(dataset['test'])), 20)
test_dataset = dataset['test'].select(test_samples)

In [7]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

# Tokenize the sampled datasets
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [8]:
# Remove unnecessary columns from the tokenized datasets
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["text"])

# Rename the 'label' column to 'labels' to match the Trainer's expectation
# [0,1]
tokenized_train_dataset = tokenized_train_dataset.rename_column("label", "labels")
tokenized_test_dataset = tokenized_test_dataset.rename_column("label", "labels")

# Set the format for PyTorch
tokenized_train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [9]:
# Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # output directory
    eval_strategy="epoch",     # evaluation strategy to adopt during training
    learning_rate=2e-5,              # learning rate
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=8,    # batch size for evaluation
    num_train_epochs=3,              # total number of training epochs
    weight_decay=0.01,               # strength of weight decay
    fp16=True ,
    report_to=["none"]
)


In [12]:
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.142456,0.800000
2,No log,0.951563,0.800000
3,No log,0.880676,0.850000


TrainOutput(global_step=75, training_loss=1.068580322265625, metrics={'train_runtime': 13.0188, 'train_samples_per_second': 46.087, 'train_steps_per_second': 5.761, 'total_flos': 79483274035200.0, 'train_loss': 1.068580322265625, 'epoch': 3.0})

In [14]:
trainer.evaluate()

{'eval_loss': 0.88067626953125,
 'eval_accuracy': 0.85,
 'eval_runtime': 0.1398,
 'eval_samples_per_second': 143.041,
 'eval_steps_per_second': 21.456,
 'epoch': 3.0}

In [18]:
import torch
import numpy as np

# 1. Set up device and move model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 2. Prepare your texts
texts = [
    "Apple is looking at buying U.K. startup for $1 billion",
    "The local team won their game last night by a large margin"
]

# 3. Tokenize and move tensors to device
tokens = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")
tokens = {k: v.to(device) for k, v in tokens.items()}

# 4. Inference (no gradient computation)
with torch.no_grad():
    outputs = model(**tokens)

# 5. Compute predictions
logits = outputs.logits        # shape (batch_size, num_labels)
preds = torch.argmax(logits, dim=-1).cpu().numpy()

# 6. Print results
for text, pred in zip(texts, preds):
    print(f"Text: {text}\nPredicted class: {pred}\n")

#AG News labels (0: World, 1: Sports, 2: Business, 3: Sci/Tech)

Text: Apple is looking at buying U.K. startup for $1 billion
Predicted class: 3

Text: The local team won their game last night by a large margin
Predicted class: 1

